# Kalman Filter Validation Against OpenRocket Sim Data

This notebook re-implements the C++ `KalmanFilter` class in Python and tests it
against exported OpenRocket simulation data. Synthetic sensor noise is injected
into the truth data to simulate realistic IMU / barometer readings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob, os

from kalman_filter import KalmanFilter

## Load OpenRocket Data

In [ ]:
sim_files = sorted(glob.glob(os.path.join("simdata", "*.csv")))
print("Found:", sim_files)

def load_sim(path):
    df = pd.read_csv(path, comment="#")
    df.columns = ["time", "alt", "vvel", "vacc"]
    return df

datasets = {os.path.basename(f).replace(".csv", ""): load_sim(f) for f in sim_files}
for name, df in datasets.items():
    print(f"{name}: {len(df)} rows, {df.time.iloc[-1]:.1f} s")

## Sensor Simulation

Generate synthetic IMU and barometer readings from the truth data:
- **IMU** runs at every sim timestep with additive Gaussian noise + a small bias.
- **Barometer** runs at a lower rate (e.g. 25 Hz) with its own noise.

In [ ]:
# tuneable sensor / filter parameters
BARO_RATE_HZ     = 50       # barometer sample rate
BARO_NOISE_STD   = 1.0      # baro noise std-dev (m)
IMU_NOISE_STD    = 0.5      # IMU accel noise std-dev (m/s^2)
IMU_BIAS_TRUE    = 0.3      # true accelerometer bias (m/s^2)

# filter tuning
STDDEV_A         = 1.0      # process noise: unmodelled accel std-dev
STDDEV_B         = 0.01     # process noise: bias drift rate std-dev
R_BARO           = BARO_NOISE_STD ** 2  # measurement noise variance

## Run Filter & Collect Results

In [ ]:
rng = np.random.default_rng(42)

results = {}  # name -> dict of arrays

for name, df in datasets.items():
    t    = df.time.values
    alt  = df.alt.values
    vvel = df.vvel.values
    vacc = df.vacc.values

    n = len(t)
    baro_period = 1.0 / BARO_RATE_HZ

    # synthetic sensor signals
    imu_readings  = vacc + IMU_BIAS_TRUE + rng.normal(0, IMU_NOISE_STD, n)
    baro_readings = alt + rng.normal(0, BARO_NOISE_STD, n)

    # init filter
    x_init = [0.0, 0.0, 0.0]
    P_init = [[10.0, 0, 0],
              [0, 10.0, 0],
              [0, 0, 1.0]]
    kf = KalmanFilter(x_init, P_init, R_BARO, STDDEV_A, STDDEV_B)

    est_h = np.zeros(n)
    est_v = np.zeros(n)
    est_b = np.zeros(n)
    baro_used = np.full(n, np.nan)  # NaN when baro not sampled

    last_baro_t = -baro_period  # force first baro update at t=0

    for i in range(n):
        dt = t[i] - t[i - 1] if i > 0 else 0.0
        if dt > 0:
            kf.predict(dt, imu_readings[i])

        # barometer update at ~BARO_RATE_HZ
        if t[i] - last_baro_t >= baro_period:
            kf.update(baro_readings[i])
            baro_used[i] = baro_readings[i]
            last_baro_t = t[i]

        est_h[i] = kf.h
        est_v[i] = kf.v
        est_b[i] = kf.b_a

    results[name] = dict(t=t, alt=alt, vvel=vvel, vacc=vacc,
                         est_h=est_h, est_v=est_v, est_b=est_b,
                         baro_used=baro_used)
    print(f"{name}  |  peak alt: truth={alt.max():.1f} m, est={est_h.max():.1f} m")

## Plots

In [ ]:
for name, r in results.items():
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    fig.suptitle(f"{name}", fontsize=14)

    # --- altitude ---
    ax = axes[0]
    ax.plot(r["t"], r["alt"], label="Truth (OpenRocket)", linewidth=1.5)
    ax.plot(r["t"], r["est_h"], label="Kalman estimate", linewidth=1.2, alpha=0.9)
    baro_mask = ~np.isnan(r["baro_used"])
    ax.scatter(r["t"][baro_mask], r["baro_used"][baro_mask],
               s=6, c="red", alpha=0.4, label="Baro readings", zorder=1)
    ax.set_ylabel("Altitude (m)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(r["t"], r["est_h"] - r["alt"], color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    # --- velocity ---
    ax = axes[1]
    ax.plot(r["t"], r["vvel"], label="Truth", linewidth=1.5)
    ax.plot(r["t"], r["est_v"], label="Kalman estimate", linewidth=1.2, alpha=0.9)
    ax.set_ylabel("Vertical velocity (m/s)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(r["t"], r["est_v"] - r["vvel"], color="gray", linewidth=0.7, alpha=0.4)
    ax2.set_ylabel("Error (m/s)", color="gray")
    ax2.tick_params(axis="y", labelcolor="gray")

    # --- bias estimate ---
    ax = axes[2]
    ax.plot(r["t"], r["est_b"], label="Bias estimate", linewidth=1.2)
    ax.axhline(IMU_BIAS_TRUE, color="k", linestyle="--", alpha=0.5, label=f"True bias ({IMU_BIAS_TRUE})")
    ax.set_ylabel("Accel bias (m/s²)")
    ax.set_xlabel("Time (s)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Error Metrics

In [ ]:
for name, r in results.items():
    alt_err = r["est_h"] - r["alt"]
    vel_err = r["est_v"] - r["vvel"]
    print(f"-------- {name}")
    print(f"  Altitude  RMSE: {np.sqrt(np.mean(alt_err**2)):.3f} m")
    print(f"  Altitude  max |err|: {np.max(np.abs(alt_err)):.3f} m\n")
    print(f"  Velocity  RMSE: {np.sqrt(np.mean(vel_err**2)):.3f} m/s")
    print(f"  Velocity  max |err|: {np.max(np.abs(vel_err)):.3f} m/s\n")
    print(f"  Final bias estimate: {r['est_b'][-1]:.4f}  (true: {IMU_BIAS_TRUE})")
    print()